<a href="https://colab.research.google.com/github/samer-glitch/TADP-Enhancing-Trustworthiness-in-Federated-Learning-Through-Provenance-Driven-Governance/blob/main/4-%20TADP-HPS-28%20Adult%20%E2%80%94%20synthetic%20documentary-evidence%20stability%20experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""TADP-HPS-28 Adult — synthetic documentary-evidence stability experiment.

Uses the archived Adult client decisions to keep the ORIGINAL 8 D2 scores,
client partitions, degradation conditions, HPS policy and five training seeds
fixed. It redraws ONLY the 20 documentary factor scores (2,000 draws by default).

Two complementary checks:
  within_tier: retain each client's original high/medium/low/missing tier,
               resample the 20 rubric levels inside that tier.
  new_profiles: resample both tier and levels using the ORIGINAL 52/33/10/5
                tier probabilities, as in the original make_manifest().

The original full training benchmark is NOT rerun or changed. No TEST data,
model fitting, or independent verification of documentary claims is involved.

Source: 1_adult_hps28.py (the original user's Adult benchmark).
Requirements: Python standard library only; works with ZIP directly.

Example:
  python TADP_Adult_Synthetic_Evidence_Stability_2000_DRAWS.py \
    --archive TADP_HPS28_Adult_Phased_Results_v5.zip \
    --draws 2000 --output TADP_Adult_Evidence_Stability_2000
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import random
import re
import sys
import statistics
import zipfile
from collections import defaultdict
from pathlib import Path

FACTORS = {
    "D1": ("source_reputation", "data_controller", "data_objective", "data_collection_lineage"),
    "D2": ("completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
           "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
           "structural_constraint_integrity"),
    "D3": ("data_dictionary", "version_logs", "collection_protocol", "definition_updates"),
    "D4": ("data_freshness", "scheduled_refresh", "retention_clarity"),
    "D5": ("regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"),
    "D6": ("license_terms", "ethical_reviews", "redistribution", "user_agreements"),
}
WEIGHTS = {"D1": .25, "D2": .15, "D3": .10, "D4": .10, "D5": .30, "D6": .10}
CRITICAL = ("D1", "D2", "D5", "D6")
D2_FACTORS = set(FACTORS["D2"])
DOC_FACTORS = set(f for dim, fs in FACTORS.items() if dim != "D2" for f in fs)
ALL_FACTORS = set.union(D2_FACTORS, DOC_FACTORS)
BANDS = {"high": (4, 5), "medium": (3, 4), "low": (1, 2), "missing": (0, 1)}
BAND_WEIGHTS = (.52, .33, .10, .05)
BASE_EVIDENCE_SEED = 7702
FL_SEEDS = (42, 43, 44, 45, 46)
ALPHAS = ("0.5", "0.1")
CONDITIONS = ("clean", "degraded")


def original_manifest_digest(manifest):
    body = {k: v for k, v in manifest.items() if k != "sha256"}
    raw = json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


def generate_docs(ids, seed, profile_map=None):
    """Replicates EXACTLY the score-drawing order in the user's make_manifest()."""
    rng = random.Random(seed)
    out = {}
    for cid in sorted(ids):
        tier = (profile_map[cid] if profile_map is not None
                else rng.choices(list(BANDS), weights=BAND_WEIGHTS)[0])
        lo, hi = BANDS[tier]
        scores = {factor: rng.randint(lo, hi) for factor in sorted(DOC_FACTORS)}
        out[cid] = {"tier": tier, "scores": scores}
    return out


def evaluate_hps(scores, policy):
    """Replicates the original evaluate() admission logic (no ablations)."""
    if set(scores) != ALL_FACTORS:
        raise ValueError("Expected 28 exact factor scores")
    if any(type(s) is not int or not (0 <= s <= 5) for s in scores.values()):
        raise ValueError("Invalid factor score")
    dims = {d: sum(scores[f] for f in fs) / len(fs) for d, fs in FACTORS.items()}
    hps = sum(WEIGHTS[d] * dims[d] for d in FACTORS)
    review = sum(WEIGHTS[d] * dims[d] for d in CRITICAL) / sum(WEIGHTS[d] for d in CRITICAL)
    if any(s == 0 for s in scores.values()):
        return False, "BLOCKED_ZERO_FACTOR", hps
    if any(value < policy["min_dimension"] for value in dims.values()):
        return False, "BLOCKED_DIMENSION_FLOOR", hps
    for factor, minimum in policy.get("critical_factor_floors", {}).items():
        if scores[factor] < minimum:
            return False, "BLOCKED_CRITICAL_FACTOR", hps
    if hps < policy["reject_below"]:
        return False, "AUTO_REJECT", hps
    if hps >= policy["direct_accept_from"]:
        return True, "AUTO_ACCEPT", hps
    review_threshold = (policy["reject_below"] + policy["direct_accept_from"]) / 2
    return (True, "REVIEW_ACCEPT", hps) if review >= review_threshold else (False, "REVIEW_REJECT", hps)


def read_archive(archive_path):
    """Load only existing decisions and original manifest, directly from ZIP."""
    pattern = re.compile(r"/adult_alpha(0\.5|0\.1)/decisions_adult_seed(4[2-6])_(clean|degraded)_none\.json$")
    by_condition = {}
    manifests = {}
    with zipfile.ZipFile(archive_path) as zf:
        paths = zf.namelist()
        for path in paths:
            m = pattern.search("/" + path.lstrip("/"))
            if m:
                key = (m.group(1), int(m.group(2)), m.group(3))
                if key in by_condition:
                    raise ValueError(f"Duplicate archived condition: {key}")
                by_condition[key] = json.loads(zf.read(path))
        for alpha in ALPHAS:
            hits = [p for p in paths if p.endswith(f"adult_alpha{alpha}/manifest_adult_K50_E7702.json")]
            if len(hits) != 1:
                raise ValueError(f"Expected one original Adult manifest for alpha={alpha}; found {len(hits)}")
            manifests[alpha] = json.loads(zf.read(hits[0]))

    expected = {(a, s, c) for a in ALPHAS for s in FL_SEEDS for c in CONDITIONS}
    if set(by_condition) != expected:
        raise ValueError(f"Missing conditions: {sorted(expected - set(by_condition))}; "
                         f"unexpected: {sorted(set(by_condition) - expected)}")
    if manifests["0.5"]["sha256"] != manifests["0.1"]["sha256"]:
        raise ValueError("Different documentary manifests across alphas; investigate before pairing")
    manifest = manifests["0.5"]
    if original_manifest_digest(manifest) != manifest["sha256"]:
        raise ValueError("Original documentary manifest hash check failed")
    return by_condition, manifest


def validate_and_prepare(archived, original_manifest):
    """Fail rather than silently changing the original experiment."""
    ids = sorted(original_manifest["clients"])
    if len(ids) != 50:
        raise ValueError(f"Expected 50 clients, found {len(ids)}")
    fixed_tiers = {cid: original_manifest["clients"][cid]["tier"] for cid in ids}
    original_docs = generate_docs(ids, BASE_EVIDENCE_SEED)
    checks = 0
    all_d2 = {}
    for key, run in sorted(archived.items()):
        if sorted(run["decisions"]) != ids:
            raise ValueError(f"Different clients for {key}")
        if run["manifest_sha256"] != original_manifest["sha256"]:
            raise ValueError(f"Original manifest hash differs for {key}")
        policy = run["policy"]
        if any(policy[k] != v for k, v in {"min_dimension": 2.5,
              "reject_below": 3.0, "direct_accept_from": 3.5}.items()):
            raise ValueError(f"Unexpected policy for {key}: {policy}")
        per_client = {}
        for cid, record in run["decisions"].items():
            archived_scores = record["factor_scores"]
            orig_scores = {f: original_manifest["clients"][cid]["documentary_factors"][f]["rubric_level"]
                           for f in DOC_FACTORS}
            if original_docs[cid]["scores"] != orig_scores:
                raise AssertionError(f"The original synthetic generator did not reproduce {cid}")
            if any(archived_scores[f] != orig_scores[f] for f in DOC_FACTORS):
                raise AssertionError(f"Documentary scores do not match manifest for {key}, {cid}")
            accepted, path, hps = evaluate_hps(archived_scores, policy)
            if accepted != record["accepted"] or path != record["path"]:
                raise AssertionError(f"Original admission decision differs for {key}, {cid}: {path}")
            if abs(hps - record["hps"]) > 1e-8:
                raise AssertionError(f"Original HPS differs for {key}, {cid}")
            per_client[cid] = {f: archived_scores[f] for f in D2_FACTORS}
            checks += 1
        all_d2[key] = per_client
    print(f"Verified {checks} original Adult client assessments and original evidence seed 7702.")
    return ids, fixed_tiers, all_d2


def write_csv(path, records, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def mean(values):
    return statistics.mean(values)


def sample_sd(values):
    return statistics.stdev(values) if len(values) >= 2 else 0.0


def run(archive_path, output_dir, n_draws=2000, seed_start=8801):
    """Fast, exact vectorized reassessment; reuses the *original* evaluation policy."""
    import numpy as np  # Available by default in Google Colab.
    if n_draws < 2:
        raise ValueError("At least two draws are needed to calculate sample SD.")
    if BASE_EVIDENCE_SEED in range(seed_start, seed_start + n_draws):
        raise ValueError("Do not include the original evidence seed 7702.")
    archived, manifest = read_archive(archive_path)
    ids, fixed_tiers, fixed_d2 = validate_and_prepare(archived, manifest)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # Each row is one ORIGINAL client assessment; 50 clients x 5 FL seeds
    # x 2 alphas x 2 conditions = 1,000 rows.
    cases = [(a, c, s, cid) for a in ALPHAS for c in CONDITIONS
             for s in FL_SEEDS for cid in ids]
    client_idx = np.asarray([ids.index(cid) for _, _, _, cid in cases], dtype=int)
    d2_matrix = np.asarray([[fixed_d2[(a, s, c)][cid][f] for f in FACTORS['D2']]
                            for a, c, s, cid in cases], dtype=float)
    d2_mean = d2_matrix.mean(axis=1)
    d2_has_zero = np.any(d2_matrix == 0, axis=1)
    old_accept = np.asarray([archived[(a, s, c)]['decisions'][cid]['accepted']
                             for a, c, s, cid in cases], dtype=bool)
    old_hps = np.asarray([archived[(a, s, c)]['decisions'][cid]['hps']
                          for a, c, s, cid in cases], dtype=float)
    groups = {(a, c): np.asarray([i for i, (a1, c1, _, _) in enumerate(cases)
                                  if a1 == a and c1 == c], dtype=int)
              for a in ALPHAS for c in CONDITIONS}
    for key in archived:
        if archived[key]['policy'].get('critical_factor_floors'):
            raise ValueError('This fast path needs a new implementation for nonempty critical-factor floors.')

    doc_names = sorted(DOC_FACTORS)
    doc_cols = {f: j for j, f in enumerate(doc_names)}
    base_count = {key: float(np.mean(old_accept[idx]) * len(ids))
                  for key, idx in groups.items()}
    result_rows = []
    # Validate the vectorization against the original scalar policy for the FIRST draw.
    verified_fast_path = False
    for mode in ('within_tier', 'new_profiles'):
        for draw_idx, evidence_seed in enumerate(range(seed_start, seed_start + n_draws)):
            docs = generate_docs(ids, evidence_seed,
                                 profile_map=fixed_tiers if mode == 'within_tier' else None)
            doc_array = np.asarray([[docs[cid]['scores'][f] for f in doc_names]
                                    for cid in ids], dtype=float)
            doc_means = {dim: np.mean(doc_array[:, [doc_cols[f] for f in fs]], axis=1)
                         for dim, fs in FACTORS.items() if dim != 'D2'}
            zero_documentary = np.any(doc_array == 0, axis=1)
            dimension_floor_fails = d2_mean < 2.5
            for dim in doc_means:
                dimension_floor_fails = dimension_floor_fails | (doc_means[dim][client_idx] < 2.5)
            zero_fails = d2_has_zero | zero_documentary[client_idx]
            hps = WEIGHTS['D2'] * d2_mean
            for dim, dim_means in doc_means.items():
                hps = hps + WEIGHTS[dim] * dim_means[client_idx]
            review = (WEIGHTS['D1'] * doc_means['D1'][client_idx] +
                      WEIGHTS['D2'] * d2_mean +
                      WEIGHTS['D5'] * doc_means['D5'][client_idx] +
                      WEIGHTS['D6'] * doc_means['D6'][client_idx]) / 0.80
            accepted = (~zero_fails & ~dimension_floor_fails &
                        ((hps >= 3.5) | ((hps >= 3.0) & (review >= 3.25))))

            if not verified_fast_path:
                for i in range(len(cases)):
                    a, c, s, cid = cases[i]
                    all_scores = {**fixed_d2[(a, s, c)][cid], **docs[cid]['scores']}
                    expected_accept, _, expected_hps = evaluate_hps(all_scores, archived[(a, s, c)]['policy'])
                    if bool(accepted[i]) != expected_accept or abs(float(hps[i]) - expected_hps) > 1e-9:
                        raise AssertionError(f'Vectorized policy does not match original for {a}, {c}, {s}, {cid}')
                print('Verified fast calculation exactly matches the original policy across 1,000 cases.')
                verified_fast_path = True

            for alpha in ALPHAS:
                for condition in CONDITIONS:
                    idx = groups[(alpha, condition)]
                    same = (accepted[idx] == old_accept[idx])
                    result_rows.append({
                        'mode': mode, 'evidence_seed': evidence_seed,
                        'alpha': alpha, 'condition': condition,
                        'baseline_admitted_mean_over_5_seeds': round(base_count[(alpha, condition)], 5),
                        'new_admitted_mean_over_5_seeds': round(float(accepted[idx].sum()) / len(FL_SEEDS), 5),
                        'decision_agreement_pct': round(100 * float(same.mean()), 5),
                        'mean_abs_hps_difference': round(float(np.mean(np.abs(hps[idx] - old_hps[idx]))), 6),
                        'accept_to_reject_occurrences': int((old_accept[idx] & ~accepted[idx]).sum()),
                        'reject_to_accept_occurrences': int((~old_accept[idx] & accepted[idx]).sum()),
                        'clients_compared': int(len(idx)),
                    })
        print(f'Finished {n_draws:,} documentary draws in mode: {mode}')

    summary_rows = []
    for mode in ('within_tier', 'new_profiles'):
        for alpha in ALPHAS:
            for condition in CONDITIONS:
                rr = [r for r in result_rows if r['mode'] == mode and
                      r['alpha'] == alpha and r['condition'] == condition]
                counts = [r['new_admitted_mean_over_5_seeds'] for r in rr]
                agreements = [r['decision_agreement_pct'] for r in rr]
                deltas = [r['mean_abs_hps_difference'] for r in rr]
                summary_rows.append({
                    'mode': mode, 'alpha': alpha, 'condition': condition,
                    'n_documentary_draws': len(rr), 'n_fl_seeds_fixed': len(FL_SEEDS),
                    'baseline_mean_admitted': rr[0]['baseline_admitted_mean_over_5_seeds'],
                    'new_mean_admitted': round(mean(counts), 4),
                    'new_sd_across_documentary_draws': round(sample_sd(counts), 4),
                    'new_min_mean_admitted': round(min(counts), 4),
                    'new_max_mean_admitted': round(max(counts), 4),
                    'decision_agreement_mean_pct': round(mean(agreements), 2),
                    'decision_agreement_sd_pct': round(sample_sd(agreements), 2),
                    'mean_abs_hps_difference': round(mean(deltas), 5),
                    'accept_to_reject_mean_per_draw': round(mean([r['accept_to_reject_occurrences'] for r in rr]), 2),
                    'reject_to_accept_mean_per_draw': round(mean([r['reject_to_accept_occurrences'] for r in rr]), 2),
                    'min_decision_agreement_pct': round(min(agreements), 2),
                    'max_decision_agreement_pct': round(max(agreements), 2),
                })
    write_csv(out / 'evidence_stability_summary.csv', summary_rows, list(summary_rows[0]))
    write_csv(out / 'evidence_stability_by_draw.csv', result_rows, list(result_rows[0]))
    (out / 'METHODS_README.txt').write_text(
        'Adult HPS-28 synthetic documentary evidence stability (no model retraining)\n'
        f'{n_draws:,} independent evidence draws per mode; evidence seeds {seed_start}–{seed_start+n_draws-1}.\n'
        'Fifty clients; original FL seeds 42–46 fixed; alphas 0.5/0.1; clean/degraded.\n'
        'within_tier: same documentary profile tier, redraw all 20 factor values.\n'
        'new_profiles: redraw profile tier from probabilities 52/33/10/5 and all 20 values.\n'
        'Same 8 measured D2 scores, same admission policy, same original archived cases.\n'
        'Each draw compares 250 historical client assessments per alpha-condition.\n'
        'Mean admitted = average admissions per FL seed; SD = across documentary draws\n'
        'of five-seed mean admissions, NOT independent training-seed SD.\n'
        'Decision agreement = agreement with original synthetic evidence assignment.\n'
        'NOT document verification accuracy, real-document generalization, or predictive accuracy.\n'
        'Increasing draws improves precision for the assumed synthetic generator only.\n',
        encoding='utf-8',
    )
    print(f'\nSynthetic evidence sensitivity ({n_draws:,} documentary draws per mode):')
    print(f'{"MODE":<14} {"alpha":<5} {"data":<9} {"original":>9} {"resampled mean±SD":>20} {"agreement":>11}')
    for row in summary_rows:
        print(f"{row['mode']:<14} {row['alpha']:<5} {row['condition']:<9} "
              f"{row['baseline_mean_admitted']:>9.1f} "
              f"{row['new_mean_admitted']:>8.2f}±{row['new_sd_across_documentary_draws']:<8.2f} "
              f"{row['decision_agreement_mean_pct']:>10.2f}%")
    print(f'\nSaved results: {out.resolve()}')
    return summary_rows

def _resolve_archive(path):
    """Find the archived Adult results ZIP; reject raw adult.csv explicitly."""
    path = Path(path)

    def accept(candidate):
        if candidate.suffix.lower() != ".zip" or not zipfile.is_zipfile(candidate):
            raise ValueError(
                "The experiment needs the ORIGINAL Adult RESULTS ZIP, not adult.csv. "
                "Upload TADP_HPS28_Adult_Phased_Results_v5.zip "
                "(containing archived decision JSON files)."
            )
        return candidate

    for candidate in (path, Path("/content") / path.name):
        if candidate.is_file():
            return accept(candidate)

    # Common case: the archive was already uploaded with a (1) filename suffix.
    # Use it only when a single unambiguous match exists.
    candidates = []
    for folder in (Path.cwd(), Path("/content")):
        if folder.is_dir():
            candidates.extend(p for p in folder.glob("TADP_HPS28_Adult_Phased_Results_v5*.zip")
                              if p.is_file() and zipfile.is_zipfile(p))
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates) == 1:
        print(f"Found existing Adult results ZIP: {candidates[0].name}")
        return candidates[0]

    try:
        import google.colab  # noqa: F401
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            "Adult results ZIP not found. Run with "
            "--archive /path/to/TADP_HPS28_Adult_Phased_Results_v5.zip. "
            "The raw adult.csv dataset is NOT used in this analysis."
        )

    for attempt in range(2):
        print("\nUPLOAD THIS FILE (not adult.csv):")
        print("  TADP_HPS28_Adult_Phased_Results_v5.zip")
        print("It contains prior HPS decisions and documentary scores.")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded
                      if name.lower().endswith(".zip") and zipfile.is_zipfile(name)]
        if len(candidates) == 1:
            return accept(candidates[0])
        print("Incorrect selection: please upload the original RESULTS .zip file. ")
        print("adult.csv is the raw dataset and cannot replace the archived decisions.")
    raise FileNotFoundError(
        "No Adult results ZIP was provided. Download or locate "
        "TADP_HPS28_Adult_Phased_Results_v5.zip and run again."
    )


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--archive", type=Path, default=Path("TADP_HPS28_Adult_Phased_Results_v5.zip"),
                        help="Existing Adult phased results ZIP; no source dataset needed")
    parser.add_argument("--output", type=Path, default=Path("TADP_Adult_Evidence_Stability"))
    parser.add_argument("--draws", type=int, default=2000,
                        help="Number of independent documentary draws for EACH mode (default: 2000)")
    parser.add_argument("--seed-start", type=int, default=8801)

    # Jupyter/Google Colab injects '-f ...kernel.json' into sys.argv.
    # Ignore ONLY that notebook-internal argument. This fixes the SystemExit: 2.
    actual_argv = list(sys.argv[1:] if argv is None else argv)
    clean_argv = []
    i = 0
    while i < len(actual_argv):
        if actual_argv[i] == "-f" and i + 1 < len(actual_argv) and actual_argv[i + 1].endswith(".json"):
            i += 2
        else:
            clean_argv.append(actual_argv[i])
            i += 1

    args = parser.parse_args(clean_argv)
    archive = _resolve_archive(args.archive)
    summary = run(archive, args.output, args.draws, args.seed_start)
    return summary


if __name__ == "__main__":
    main()


Verified 1000 original Adult client assessments and original evidence seed 7702.
Verified fast calculation exactly matches the original policy across 1,000 cases.
Finished 2,000 documentary draws in mode: within_tier
Finished 2,000 documentary draws in mode: new_profiles

Synthetic evidence sensitivity (2,000 documentary draws per mode):
MODE           alpha data       original    resampled mean±SD   agreement
within_tier    0.5   clean          30.2    30.20±0.02         100.00%
within_tier    0.5   degraded       29.4    29.54±0.14          99.49%
within_tier    0.1   clean          16.2    16.40±0.03          99.60%
within_tier    0.1   degraded       14.6    14.78±0.08          99.56%
new_profiles   0.5   clean          30.2    30.96±1.90          80.39%
new_profiles   0.5   degraded       29.4    30.23±1.86          80.58%
new_profiles   0.1   clean          16.2    16.32±1.11          90.00%
new_profiles   0.1   degraded       14.6    14.75±1.02          90.77%

Saved results: /c